# 02. Variáveis e alvo

**Objetivo:** transformar as compras em uma tabela com **uma linha por cliente**, pronta para o modelo.

**A ideia central:** escolhemos uma **data de corte**.

- **O passado** (antes do corte) gera as variáveis que descrevem o cliente.
- **O futuro** (3 meses depois do corte) gera a resposta que queremos prever: o cliente comprou de novo?

Assim o modelo nunca "vê" o futuro ao aprender, que é o que acontece na vida real.

In [1]:
import pandas as pd

purchases = pd.read_csv('../data/processed/compras.csv', parse_dates=['InvoiceDate'])
print(f'{len(purchases):,} purchases')

407,664 purchases


## 1. Função que monta a tabela de clientes

Para cada cliente que já tinha comprado **antes do corte**, calculamos:

| Variável | O que significa |
|---|---|
| `recency_days` | dias desde a última compra até o corte |
| `invoice_count` | quantas faturas diferentes (frequência) |
| `total_quantity` | quantos itens comprou |
| `total_spend` | quanto gastou |
| `repurchased` | **alvo**: 1 se comprou entre o corte e a data final, 0 se não |

In [2]:
def build_customer_table(purchases, corte, fim):
    corte, fim = pd.Timestamp(corte), pd.Timestamp(fim)

    past_data = purchases[purchases['InvoiceDate'] < corte]
    future_data = purchases[(purchases['InvoiceDate'] >= corte) & (purchases['InvoiceDate'] < fim)]

    # checagem contra vazamento: nada do past_data pode ser posterior ao corte
    assert past_data['InvoiceDate'].max() < corte

    customer_table = past_data.groupby('CustomerID').agg(
        ultima_compra=('InvoiceDate', 'max'),
        invoice_count=('Invoice', 'nunique'),
        total_quantity=('Quantity', 'sum'),
        total_spend=('line_total', 'sum'),
    )
    customer_table['recency_days'] = (corte - customer_table['ultima_compra']).dt.days
    customer_table = customer_table.drop(columns='ultima_compra')

    # alvo: o cliente aparece em alguma compra do período future_data?
    customer_table['repurchased'] = customer_table.index.isin(future_data['CustomerID']).astype(int)
    return customer_table

## 2. Dois cortes: um para treinar, outro para testar

- **Treino:** corte em **09/06/2010**, olhando as compras até **09/09/2010**.
- **Teste:** corte em **09/09/2010**, olhando as compras até **10/12/2010**.

**Por que assim?** O período de teste vem **depois** do de treino e não se sobrepõe a ele: o modelo aprende com o passado e é avaliado no futuro, como seria usado de verdade. Se misturássemos clientes ao acaso entre treino e teste, o modelo poderia ser avaliado em um período que já conhece.

In [3]:
train_data = build_customer_table(purchases, corte='2010-06-09', fim='2010-09-09')
test_data = build_customer_table(purchases, corte='2010-09-09', fim='2010-12-10')

print(f'Treino: {len(train_data):,} clientes | recompraram: {train_data["repurchased"].mean():.1%}')
print(f'Teste:  {len(test_data):,} clientes | recompraram: {test_data["repurchased"].mean():.1%}')

Treino: 2,780 clientes | recompraram: 48.7%
Teste:  3,359 clientes | recompraram: 58.1%


**Atenção a este ponto:** a taxa de recompra costuma ser diferente nos dois períodos. A janela de teste (setembro a dezembro) inclui o fim de ano, época de mais vendas, enquanto a de treino (junho a setembro) não. Isso é uma limitação e deve aparecer na conclusão do projeto.

In [4]:
display(train_data.head())
print('\nResumo das variáveis no train_data:')
display(train_data.describe().T.round(1))

,invoice_count,total_quantity,total_spend,recency_days,repurchased
CustomerID,,,,,
12346,10,51,230.55,98,1
12349,2,474,1268.52,21,0
12355,1,303,488.21,18,0
12358,2,590,1697.93,1,0
12359,4,777,1522.23,77,1



Resumo das variáveis no train_data:


,count,mean,std,min,25%,50%,75%,max
invoice_count,2780.0,3.0,4.7,1.0,1.0,2.0,3.0,75.0
total_quantity,2780.0,898.0,5559.6,1.0,121.0,285.0,670.2,220600.0
total_spend,2780.0,1377.6,5402.3,1.3,252.9,526.0,1156.6,165462.2
recency_days,2780.0,62.2,53.0,0.0,19.0,46.0,95.0,189.0
repurchased,2780.0,0.5,0.5,0.0,0.0,0.0,1.0,1.0


In [5]:
train_data.to_csv('../data/processed/treino.csv')
test_data.to_csv('../data/processed/teste.csv')
print('Tabelas salvas em data/processed/')

Tabelas salvas em data/processed/


## Conclusões

- A tabela de treino contém **2.780 clientes**, dos quais **48,7% (1.353)** recompraram nos três meses seguintes ao corte.
- A tabela de teste contém **3.359 clientes**, dos quais **58,1% (1.951)** recompraram no período futuro observado.
- A taxa de recompra no teste foi **9,4 pontos percentuais maior** que no treino. Uma explicação possível é a sazonalidade: o período de teste inclui outubro, novembro e início de dezembro, quando houve mais faturas. Isso é uma hipótese plausível, não uma prova de que a sazonalidade explica toda a diferença.
- A separação temporal é mais próxima do uso real do modelo do que uma divisão aleatória, mas os períodos são diferentes. Portanto, os resultados devem ser interpretados com cautela.
- As variáveis de entrada (`recency_days`, `invoice_count`, `total_quantity` e `total_spend`) são calculadas somente com dados anteriores ao corte; `repurchased` representa a recompra no período futuro.